<a href="https://colab.research.google.com/github/AngelDavidRuizB/AngelDavidRuizB/blob/main/Ejercicios_semana_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Ejercicio 2

In [ ]:

import random, copy

TRUTH_TABLE = {
    (0,0,0,0): (1,1,1,1,1,1,0),
    (0,0,0,1): (0,1,1,0,0,0,0),
    (0,0,1,0): (1,1,0,1,1,0,1),
    (0,0,1,1): (1,1,1,1,0,0,1),
    (0,1,0,0): (0,1,1,0,0,1,1),
    (0,1,0,1): (1,0,1,1,0,1,1),
    (0,1,1,0): (1,0,1,1,1,1,1),
    (0,1,1,1): (1,1,1,0,0,0,0),
    (1,0,0,0): (1,1,1,1,1,1,1),
    (1,0,0,1): (1,1,1,1,0,1,1),
}

TERMINALS = ['A','B','C','D','nA','nB','nC','nD']
FUNCTIONS = {
    'AND':  {'arity':2,'func':lambda x,y:int(x and y)},
    'OR':   {'arity':2,'func':lambda x,y:int(x or y)},
    'NOT':  {'arity':1,'func':lambda x:int(not x)},
    'XOR':  {'arity':2,'func':lambda x,y:int(x!=y)},
    'NAND': {'arity':2,'func':lambda x,y:int(not(x and y))},
    'NOR':  {'arity':2,'func':lambda x,y:int(not(x or y))},
}

class Node:
    def __init__(self, value, children=None):
        self.value = value
        self.children = children or []
    def is_terminal(self): return self.value in TERMINALS
    def copy(self): return copy.deepcopy(self)
    def count_nodes(self):
        return 1 + sum(c.count_nodes() for c in self.children)
    def to_string(self):
        if self.is_terminal(): return self.value
        if len(self.children)==1: return f"{self.value}({self.children[0].to_string()})"
        return f"{self.value}({','.join(c.to_string() for c in self.children)})"

def gen_tree(max_depth=4, depth=0):
    if depth>=max_depth or (depth>0 and random.random()<0.35):
        return Node(random.choice(TERMINALS))
    fname = random.choice(list(FUNCTIONS.keys()))
    ar = FUNCTIONS[fname]['arity']
    return Node(fname, [gen_tree(max_depth, depth+1) for _ in range(ar)])

def eval_tree(node, inp):
    A,B,C,D = inp
    env = {'A':A,'B':B,'C':C,'D':D,'nA':1-A,'nB':1-B,'nC':1-C,'nD':1-D}
    def ev(n):
        if n.is_terminal(): return env[n.value]
        return FUNCTIONS[n.value]['func'](*[ev(c) for c in n.children])
    return ev(node)

def fitness(tree, seg):
    err = sum(1 for inp,out in TRUTH_TABLE.items() if eval_tree(tree,inp)!=out[seg])
    return 1.0/(0.1+err)

def get_node(tree, idx):
    ctr=[0]; res=[None]
    def tr(n):
        if res[0]: return
        if ctr[0]==idx: res[0]=n
        ctr[0]+=1
        for c in n.children: tr(c)
    tr(tree); return res[0]

def set_node(tree, idx, sub):
    ctr=[0]; done=[False]
    def tr(n):
        if done[0]: return
        for i,c in enumerate(n.children):
            ctr[0]+=1
            if ctr[0]==idx: n.children[i]=sub; done[0]=True; return
            tr(c)
    tr(tree)

def crossover(t1, t2):
    a,b = t1.copy(), t2.copy()
    n1,n2 = a.count_nodes(), b.count_nodes()
    if n1<=1 or n2<=1: return a,b
    c1,c2 = random.randint(1,n1-1), random.randint(1,n2-1)
    s1 = get_node(a,c1).copy(); s2 = get_node(b,c2).copy()
    set_node(a,c1,s2); set_node(b,c2,s1)
    return a,b

def mutate(tree):
    t = tree.copy(); n = t.count_nodes()
    mp = random.randint(0,n-1); nd = get_node(t,mp)
    if nd:
        if nd.is_terminal(): nd.value = random.choice(TERMINALS)
        else:
            ar = FUNCTIONS[nd.value]['arity']
            cands = [f for f,v in FUNCTIONS.items() if v['arity']==ar]
            if cands: nd.value = random.choice(cands)
    return t

def roulette(probs):
    r=random.random(); cum=0
    for i,p in enumerate(probs):
        cum+=p
        if r<=cum: return i
    return len(probs)-1

def run_gp(seg_idx, seg_name, pop_size=80, gens=150, pc=0.7, pm=0.01):
    pop = [gen_tree(4) for _ in range(pop_size)]
    best_t, best_f = None, -1
    print(f"  Segmento {seg_name}: ", end='', flush=True)
    for g in range(gens):
        fits = [fitness(ind,seg_idx) for ind in pop]
        mf = max(fits)
        if mf > best_f:
            best_f = mf
            best_t = pop[fits.index(mf)].copy()
        if best_f >= 10.0: print(f"PERFECTO en gen {g}"); break
        tf = sum(fits)+1e-10; probs=[f/tf for f in fits]
        srt = sorted(zip(fits,pop),key=lambda x:x[0],reverse=True)
        new_pop = [srt[0][1].copy(), srt[1][1].copy()]
        while len(new_pop)<pop_size:
            p1=pop[roulette(probs)]; p2=pop[roulette(probs)]
            c1,c2 = (crossover(p1,p2) if random.random()<pc else (p1.copy(),p2.copy()))
            if random.random()<pm: c1=mutate(c1)
            if random.random()<pm: c2=mutate(c2)
            if c1.count_nodes()<60: new_pop.append(c1)
            if c2.count_nodes()<60 and len(new_pop)<pop_size: new_pop.append(c2)
        pop = new_pop
    else:
        print(f"f_apt={best_f:.4f}")
    return best_t, best_f

if __name__=="__main__":
    random.seed(42)
    print("="*60)
    print("EJERCICIO 2: PG - CODIFICADOR BCD A 7 SEGMENTOS")
    print("="*60)
    print("Terminales T = {A, B, C, D, nA, nB, nC, nD}")
    print("Funciones  F = {AND, OR, NOT, XOR, NAND, NOR}")
    print("Aptitud:  f_apt = 1 / (0.1 + errores)")
    print()
    segs = ['a','b','c','d','e','f','g']
    results = {}
    for i,s in enumerate(segs):
        t,f = run_gp(i,s)
        results[s] = (t,f)
    print("\n--- RESULTADOS ---")
    total_acc = 0
    for i,s in enumerate(segs):
        t,f = results[s]
        errs = sum(1 for inp,out in TRUTH_TABLE.items() if eval_tree(t,inp)!=out[i])
        acc = (10-errs)/10*100
        total_acc += acc
        expr = t.to_string()[:80]
        print(f"  Seg {s}: {acc:.0f}% correcto | f_apt={f:.3f}")
        print(f"    Expr: {expr}")
    print(f"\nPrecision promedio: {total_acc/7:.1f}%")

EJERCICIO 2: PG - CODIFICADOR BCD A 7 SEGMENTOS
Terminales T = {A, B, C, D, nA, nB, nC, nD}
Funciones  F = {AND, OR, NOT, XOR, NAND, NOR}
Aptitud:  f_apt = 1 / (0.1 + errores)

  Segmento a: 

KeyboardInterrupt: 

## Ejercicio 3

In [ ]:

import random, copy, math

# ============================================================
# DEFINICIÓN DEL PROBLEMA
# ============================================================
# Sala cuadrada de GRID_SIZE x GRID_SIZE
# Ingenieros en posiciones aleatorias (fijas por semilla)
# Robot inicia en (0,0) y realiza MAX_STEPS movimientos
# Cada vez que el robot llega a la posición de un ingeniero: +10 puntos
# El robot tiene sensores: distancia a ingenieros cercanos

GRID_SIZE = 10
MAX_STEPS = 50
NUM_ENGINEERS = 5

random.seed(123)
ENGINEERS = [(random.randint(0,GRID_SIZE-1), random.randint(0,GRID_SIZE-1))
             for _ in range(NUM_ENGINEERS)]
random.seed(None)  # restaurar aleatoriedad

print(f"Sala: {GRID_SIZE}x{GRID_SIZE} | Ingenieros: {ENGINEERS}")

# ============================================================
# CONJUNTO DE TERMINALES T
# ============================================================
# Sensores del robot (entradas al programa):
#   dx_near: desplazamiento x al ingeniero más cercano (-1,0,1)
#   dy_near: desplazamiento y al ingeniero más cercano (-1,0,1)
#   dist_near: distancia Manhattan al más cercano (0..10)
#   cookies_left: galletas restantes (0..5)
#   rand_val: valor aleatorio (exploración)
TERMINALS = ['dx_near', 'dy_near', 'dist_near', 'cookies_left', 'rand_val', '1', '-1', '0']

# ============================================================
# CONJUNTO DE FUNCIONES F
# ============================================================
# Funciones numéricas y de decisión:
#   ADD(a,b), SUB(a,b), MUL(a,b), IFLTE(a,b,c,d): if a<=b then c else d
#   SIGN(x): -1,0,1, MAX(a,b), MIN(a,b)
def safe_mul(a,b):
    r = a*b
    return max(-10, min(10, r))

def iflte(a,b,c,d):  # if a <= b then c else d
    return c if a<=b else d

FUNCTIONS = {
    'ADD':   {'arity':2,'func':lambda a,b: max(-10,min(10,a+b))},
    'SUB':   {'arity':2,'func':lambda a,b: max(-10,min(10,a-b))},
    'MUL':   {'arity':2,'func':safe_mul},
    'IFLTE': {'arity':4,'func':iflte},
    'SIGN':  {'arity':1,'func':lambda x: (1 if x>0 else (-1 if x<0 else 0))},
    'MAX':   {'arity':2,'func':max},
    'MIN':   {'arity':2,'func':min},
    'ABS':   {'arity':1,'func':abs},
}

# ============================================================
# ÁRBOL DE EXPRESIÓN
# ============================================================
class Node:
    def __init__(self, value, children=None):
        self.value = value
        self.children = children or []
    def is_terminal(self): return self.value in TERMINALS
    def copy(self): return copy.deepcopy(self)
    def count_nodes(self):
        return 1 + sum(c.count_nodes() for c in self.children)
    def to_string(self):
        if self.is_terminal(): return str(self.value)
        return f"{self.value}({','.join(c.to_string() for c in self.children)})"

def gen_tree(max_depth=5, depth=0):
    if depth>=max_depth or (depth>1 and random.random()<0.4):
        return Node(random.choice(TERMINALS))
    fname = random.choice(list(FUNCTIONS.keys()))
    ar = FUNCTIONS[fname]['arity']
    return Node(fname,[gen_tree(max_depth,depth+1) for _ in range(ar)])

def eval_tree(node, env):
    if node.is_terminal():
        try: return float(node.value)
        except: return env.get(node.value, 0.0)
    args = [eval_tree(c,env) for c in node.children]
    try: return FUNCTIONS[node.value]['func'](*args)
    except: return 0.0

# ============================================================
# SIMULACIÓN DEL ROBOT (EJECUCIÓN DEL PROGRAMA)
# ============================================================
def get_nearest_engineer(rx, ry, remaining_engineers):
    """Encuentra el ingeniero más cercano y retorna info de dirección"""
    if not remaining_engineers:
        return 0, 0, 99
    dists = [(abs(rx-ex)+abs(ry-ey), ex, ey) for ex,ey in remaining_engineers]
    dists.sort()
    d, ex, ey = dists[0]
    dx = 1 if ex>rx else (-1 if ex<rx else 0)
    dy = 1 if ey>ry else (-1 if ey<ry else 0)
    return dx, dy, d

def simulate_robot(tree_x, tree_y, verbose=False):
    """
    Simula el recorrido del robot.
    tree_x: árbol que decide el movimiento en X
    tree_y: árbol que decide el movimiento en Y
    Retorna puntos totales obtenidos.
    """
    rx, ry = 0, 0
    remaining = list(ENGINEERS)
    total_points = 0
    path = [(rx, ry)]

    for step in range(MAX_STEPS):
        if not remaining:
            break

        dx_near, dy_near, dist_near = get_nearest_engineer(rx, ry, remaining)

        env = {
            'dx_near': float(dx_near),
            'dy_near': float(dy_near),
            'dist_near': float(dist_near),
            'cookies_left': float(len(remaining)),
            'rand_val': random.uniform(-1, 1),
            '1': 1.0, '-1': -1.0, '0': 0.0
        }

        # Evaluar árboles para obtener movimiento
        mx = eval_tree(tree_x, env)
        my = eval_tree(tree_y, env)

        # Convertir a dirección discreta (-1, 0, 1)
        move_x = 1 if mx > 0.3 else (-1 if mx < -0.3 else 0)
        move_y = 1 if my > 0.3 else (-1 if my < -0.3 else 0)

        # Aplicar movimiento (dentro de los límites de la sala)
        rx = max(0, min(GRID_SIZE-1, rx + move_x))
        ry = max(0, min(GRID_SIZE-1, ry + move_y))
        path.append((rx, ry))

        # Verificar si el robot llegó a un ingeniero
        for eng in remaining[:]:
            if (rx, ry) == eng:
                total_points += 10
                remaining.remove(eng)
                if verbose:
                    print(f"  Paso {step}: Robot en {(rx,ry)} entrega galleta! Puntos={total_points}")

    return total_points, path

# ============================================================
# FUNCIÓN DE APTITUD
# ============================================================
def fitness_robot(tree_x, tree_y, runs=5):
    """
    f_apt = promedio de puntos en 'runs' simulaciones
    (múltiples corridas para reducir varianza por rand_val)
    """
    total = sum(simulate_robot(tree_x, tree_y)[0] for _ in range(runs))
    return total / runs

# ============================================================
# OPERADORES GENÉTICOS (igual que ejercicio 2)
# ============================================================
def get_node(tree,idx):
    ctr=[0];res=[None]
    def tr(n):
        if res[0]: return
        if ctr[0]==idx: res[0]=n
        ctr[0]+=1
        for c in n.children: tr(c)
    tr(tree); return res[0]

def set_node(tree,idx,sub):
    ctr=[0];done=[False]
    def tr(n):
        if done[0]: return
        for i,c in enumerate(n.children):
            ctr[0]+=1
            if ctr[0]==idx: n.children[i]=sub;done[0]=True;return
            tr(c)
    tr(tree)

def crossover(t1,t2):
    a,b=t1.copy(),t2.copy()
    n1,n2=a.count_nodes(),b.count_nodes()
    if n1<=1 or n2<=1: return a,b
    c1,c2=random.randint(1,n1-1),random.randint(1,n2-1)
    s1=get_node(a,c1).copy();s2=get_node(b,c2).copy()
    set_node(a,c1,s2);set_node(b,c2,s1)
    return a,b

def mutate(tree):
    t=tree.copy();n=t.count_nodes()
    mp=random.randint(0,n-1);nd=get_node(t,mp)
    if nd:
        if nd.is_terminal(): nd.value=random.choice(TERMINALS)
        else:
            ar=FUNCTIONS[nd.value]['arity']
            cands=[f for f,v in FUNCTIONS.items() if v['arity']==ar]
            if cands: nd.value=random.choice(cands)
    return t

def roulette(probs):
    r=random.random();cum=0
    for i,p in enumerate(probs):
        cum+=p
        if r<=cum: return i
    return len(probs)-1

# ============================================================
# ALGORITMO PG PARA EL ROBOT (COEVOLUCIÓN DE 2 ÁRBOLES)
# ============================================================
def run_gp_robot(pop_size=60, gens=100, pc=0.7, pm=0.05):
    """
    Evoluciona pares (tree_x, tree_y) simultáneamente.
    Cada individuo es una tupla (árbol_movimiento_x, árbol_movimiento_y).
    """
    # Población inicial: pares de árboles
    population = [(gen_tree(4), gen_tree(4)) for _ in range(pop_size)]
    best_pair = None
    best_fit = -1
    history = []

    print(f"\nEjecutando PG para robot ({gens} generaciones)...")

    for g in range(gens):
        fits = [fitness_robot(tx, ty, runs=3) for tx,ty in population]
        mf = max(fits)
        history.append(mf)

        if mf > best_fit:
            best_fit = mf
            best_pair = (population[fits.index(mf)][0].copy(),
                         population[fits.index(mf)][1].copy())

        if g % 20 == 0:
            print(f"  Gen {g:3d}: mejor aptitud = {best_fit:.1f} puntos")

        # Parada si puntuación perfecta (50 pts = 5 ingenieros * 10)
        if best_fit >= 50.0:
            print(f"  Gen {g}: SOLUCIÓN ÓPTIMA encontrada!")
            break

        tf=sum(fits)+1e-10; probs=[f/tf for f in fits]
        srt=sorted(zip(fits,population),key=lambda x:x[0],reverse=True)
        new_pop=[(srt[0][1][0].copy(),srt[0][1][1].copy()),
                 (srt[1][1][0].copy(),srt[1][1][1].copy())]

        while len(new_pop)<pop_size:
            p1=population[roulette(probs)]
            p2=population[roulette(probs)]
            # Cruce independiente para X e Y
            if random.random()<pc:
                cx1,cx2=crossover(p1[0],p2[0])
                cy1,cy2=crossover(p1[1],p2[1])
            else:
                cx1,cy1=p1[0].copy(),p1[1].copy()
                cx2,cy2=p2[0].copy(),p2[1].copy()
            if random.random()<pm: cx1=mutate(cx1)
            if random.random()<pm: cy1=mutate(cy1)
            if random.random()<pm: cx2=mutate(cx2)
            if random.random()<pm: cy2=mutate(cy2)
            if cx1.count_nodes()<60 and cy1.count_nodes()<60:
                new_pop.append((cx1,cy1))
            if cx2.count_nodes()<60 and cy2.count_nodes()<60 and len(new_pop)<pop_size:
                new_pop.append((cx2,cy2))
        population=new_pop

    return best_pair, best_fit, history

# ============================================================
# EJECUCIÓN Y VISUALIZACIÓN
# ============================================================
if __name__=="__main__":
    random.seed(42)
    print("="*60)
    print("EJERCICIO 3: PG - ROBOT REPARTIDOR DE GALLETAS")
    print("="*60)
    print(f"\nConfiguracion:")
    print(f"  Sala: {GRID_SIZE}x{GRID_SIZE} unidades")
    print(f"  Ingenieros: {NUM_ENGINEERS} personas en {ENGINEERS}")
    print(f"  Pasos maximos: {MAX_STEPS}")
    print(f"  Puntos por galleta: 10")
    print(f"\nConjunto de Terminales T =")
    print(f"  dx_near: dir X al ing. mas cercano (-1,0,1)")
    print(f"  dy_near: dir Y al ing. mas cercano (-1,0,1)")
    print(f"  dist_near: distancia Manhattan al mas cercano")
    print(f"  cookies_left: galletas restantes")
    print(f"  rand_val: componente aleatoria (exploracion)")
    print(f"  1, -1, 0: constantes")
    print(f"\nConjunto de Funciones F =")
    print(f"  ADD, SUB, MUL, IFLTE (if<=then/else), SIGN, MAX, MIN, ABS")
    print(f"\nFuncion de Aptitud:")
    print(f"  f_apt = promedio de puntos en 3 simulaciones")
    print(f"  Maximo posible = {NUM_ENGINEERS*10} puntos")

    (best_tx, best_ty), best_fit, history = run_gp_robot(
        pop_size=60, gens=100, pc=0.7, pm=0.05)

    print(f"\n--- MEJOR SOLUCION ENCONTRADA ---")
    print(f"Aptitud = {best_fit:.1f} / {NUM_ENGINEERS*10} puntos")
    print(f"Arbol movimiento X: {best_tx.to_string()[:80]}")
    print(f"Arbol movimiento Y: {best_ty.to_string()[:80]}")

    print(f"\n--- SIMULACION DETALLADA DEL MEJOR ROBOT ---")
    random.seed(0)
    pts, path = simulate_robot(best_tx, best_ty, verbose=True)
    print(f"\nPuntos totales: {pts}")
    print(f"Trayectoria ({len(path)} pasos): {path[:15]}{'...' if len(path)>15 else ''}")

    # Visualización ASCII de la sala
    print(f"\n--- MAPA DE LA SALA ({GRID_SIZE}x{GRID_SIZE}) ---")
    grid = [['.' for _ in range(GRID_SIZE)] for _ in range(GRID_SIZE)]
    for i,(ex,ey) in enumerate(ENGINEERS):
        grid[ey][ex] = str(i+1)
    # Marcar trayectoria
    for (px,py) in path[1:]:
        if grid[py][px]=='.': grid[py][px]='*'
    # Inicio y fin
    grid[path[0][1]][path[0][0]] = 'S'
    grid[path[-1][1]][path[-1][0]] = 'E'
    print("  " + " ".join(str(i) for i in range(GRID_SIZE)))
    for row_idx,row in enumerate(grid):
        print(f"{row_idx} " + " ".join(row))
    print("  S=Inicio E=Fin 1-5=Ingenieros *=Trayectoria")

## Ejercicio 4

In [ ]:
"""
EJERCICIO 4 - Programación Genética para Control
Basado en: https://www.youtube.com/watch?v=6KNuJn6dVy4
Ejemplo: Control de un péndulo invertido (balancín sobre carro)
El programa PG controla la fuerza aplicada al carro para mantener el péndulo en equilibrio.
"""
import random, copy, math

# ============================================================
# DEFINICIÓN DEL SISTEMA FÍSICO: PÉNDULO INVERTIDO
# ============================================================
# Estado: [x, x_dot, theta, theta_dot]
#   x      = posición del carro
#   x_dot  = velocidad del carro
#   theta  = ángulo del péndulo (0 = vertical, meta = 0)
#   theta_dot = velocidad angular

# Parámetros físicos
g   = 9.8    # gravedad (m/s^2)
mc  = 1.0    # masa carro (kg)
mp  = 0.1    # masa péndulo (kg)
L   = 0.5    # longitud péndulo (m)
dt  = 0.02   # paso de tiempo (s)
MAX_T = 500  # pasos máximos de simulación (~10 segundos)

# Límites de estabilidad
X_LIMIT     = 2.4   # ±2.4 m
THETA_LIMIT = math.pi/6  # ±30 grados (falla si supera esto)

def step_physics(state, force):
    """Simulación física del péndulo invertido (ecuaciones de Newton)"""
    x, xd, th, thd = state
    # Ecuaciones de movimiento (modelo estándar péndulo sobre carro)
    sin_th = math.sin(th)
    cos_th = math.cos(th)
    temp = (force + mp*L*thd**2*sin_th) / (mc + mp)
    th_acc = (g*sin_th - cos_th*temp) / (L*(4/3 - mp*cos_th**2/(mc+mp)))
    x_acc  = temp - mp*L*th_acc*cos_th/(mc+mp)
    # Integración Euler
    xd_new  = xd  + dt*x_acc
    x_new   = x   + dt*xd_new
    thd_new = thd + dt*th_acc
    th_new  = th  + dt*thd_new
    return [x_new, xd_new, th_new, thd_new]

def is_failed(state):
    x, xd, th, thd = state
    return abs(x)>X_LIMIT or abs(th)>THETA_LIMIT

# ============================================================
# CONJUNTO DE TERMINALES T
# ============================================================
# Sensores del controlador (lecturas del estado del sistema):
#   pos_x     : posición del carro (-2.4 a 2.4 m)
#   vel_x     : velocidad del carro
#   angle     : ángulo del péndulo (rad)
#   ang_vel   : velocidad angular del péndulo
#   sin_angle : sen(ángulo) - muy útil para control
#   cos_angle : cos(ángulo)
TERMINALS = ['pos_x', 'vel_x', 'angle', 'ang_vel',
             'sin_angle', 'cos_angle', '1.0', '-1.0', '0.5', '-0.5']

# ============================================================
# CONJUNTO DE FUNCIONES F
# ============================================================
# Funciones aritméticas y condicionales para generar señales de control:
def protected_div(a, b):
    return a/b if abs(b)>1e-6 else 1.0

def iflte(a,b,c,d): return c if a<=b else d
def clamp10(x): return max(-10.0, min(10.0, x))

FUNCTIONS = {
    'ADD':   {'arity':2,'func':lambda a,b: clamp10(a+b)},
    'SUB':   {'arity':2,'func':lambda a,b: clamp10(a-b)},
    'MUL':   {'arity':2,'func':lambda a,b: clamp10(a*b)},
    'DIV':   {'arity':2,'func':protected_div},
    'IFLTE': {'arity':4,'func':iflte},
    'NEG':   {'arity':1,'func':lambda x:-x},
    'ABS':   {'arity':1,'func':abs},
    'MAX':   {'arity':2,'func':max},
    'MIN':   {'arity':2,'func':min},
}

# ============================================================
# ÁRBOL DE EXPRESIÓN
# ============================================================
class Node:
    def __init__(self, value, children=None):
        self.value=value; self.children=children or []
    def is_terminal(self): return self.value in TERMINALS
    def copy(self): return copy.deepcopy(self)
    def count_nodes(self):
        return 1+sum(c.count_nodes() for c in self.children)
    def to_string(self):
        if self.is_terminal(): return str(self.value)
        return f"{self.value}({','.join(c.to_string() for c in self.children)})"

def gen_tree(max_depth=5, depth=0):
    if depth>=max_depth or (depth>1 and random.random()<0.35):
        return Node(random.choice(TERMINALS))
    fname=random.choice(list(FUNCTIONS.keys()))
    ar=FUNCTIONS[fname]['arity']
    return Node(fname,[gen_tree(max_depth,depth+1) for _ in range(ar)])

def eval_tree(node, env):
    if node.is_terminal():
        try: return float(node.value)
        except: return env.get(node.value,0.0)
    args=[eval_tree(c,env) for c in node.children]
    try: return FUNCTIONS[node.value]['func'](*args)
    except: return 0.0

# ============================================================
# FUNCIÓN DE APTITUD
# ============================================================
def fitness_control(tree, n_trials=3):
    """
    f_apt = pasos promedio que el péndulo se mantiene en equilibrio.
    Se premia el tiempo de equilibrio y se penaliza la desviación.
    Se usan múltiples condiciones iniciales para robustez.
    """
    init_conditions = [
        [0.0, 0.0,  0.05, 0.0],   # pequeño ángulo positivo
        [0.0, 0.0, -0.05, 0.0],   # pequeño ángulo negativo
        [0.1, 0.0,  0.02, 0.0],   # desplazamiento lateral
    ]
    total_steps = 0
    total_reward = 0

    for init in init_conditions:
        state = list(init)
        for step in range(MAX_T):
            x, xd, th, thd = state
            env = {
                'pos_x':     x,
                'vel_x':     xd,
                'angle':     th,
                'ang_vel':   thd,
                'sin_angle': math.sin(th),
                'cos_angle': math.cos(th),
                '1.0': 1.0, '-1.0': -1.0,
                '0.5': 0.5, '-0.5': -0.5
            }
            # El árbol genera la fuerza de control
            force = eval_tree(tree, env)
            force = max(-10.0, min(10.0, force))  # saturar la fuerza

            state = step_physics(state, force)

            if is_failed(state):
                break
            total_steps += 1
            # Recompensa adicional por mantener ángulo pequeño
            total_reward += 1.0 - abs(state[2])/THETA_LIMIT

    # f_apt = (pasos + recompensa_ángulo) / condiciones_iniciales
    return (total_steps + 0.1*total_reward) / len(init_conditions)

# ============================================================
# OPERADORES GENÉTICOS
# ============================================================
def get_node(tree,idx):
    ctr=[0];res=[None]
    def tr(n):
        if res[0]: return
        if ctr[0]==idx: res[0]=n
        ctr[0]+=1
        for c in n.children: tr(c)
    tr(tree); return res[0]

def set_node(tree,idx,sub):
    ctr=[0];done=[False]
    def tr(n):
        if done[0]: return
        for i,c in enumerate(n.children):
            ctr[0]+=1
            if ctr[0]==idx: n.children[i]=sub;done[0]=True;return
            tr(c)
    tr(tree)

def crossover(t1,t2):
    a,b=t1.copy(),t2.copy()
    n1,n2=a.count_nodes(),b.count_nodes()
    if n1<=1 or n2<=1: return a,b
    c1,c2=random.randint(1,n1-1),random.randint(1,n2-1)
    s1=get_node(a,c1).copy();s2=get_node(b,c2).copy()
    set_node(a,c1,s2);set_node(b,c2,s1)
    return a,b

def mutate(tree):
    t=tree.copy();n=t.count_nodes()
    mp=random.randint(0,n-1);nd=get_node(t,mp)
    if nd:
        if nd.is_terminal(): nd.value=random.choice(TERMINALS)
        else:
            ar=FUNCTIONS[nd.value]['arity']
            cands=[f for f,v in FUNCTIONS.items() if v['arity']==ar]
            if cands: nd.value=random.choice(cands)
    return t

def roulette(probs):
    r=random.random();cum=0
    for i,p in enumerate(probs):
        cum+=p
        if r<=cum: return i
    return len(probs)-1

# ============================================================
# ALGORITMO PG PARA CONTROL
# ============================================================
def run_gp_control(pop_size=80, gens=150, pc=0.7, pm=0.02):
    pop = [gen_tree(5) for _ in range(pop_size)]
    best_t, best_f = None, -1
    history = []

    print(f"\nEjecutando PG para control de pendulo ({gens} gens)...")
    for g in range(gens):
        fits = [fitness_control(ind) for ind in pop]
        mf = max(fits)
        history.append(mf)

        if mf > best_f:
            best_f = mf
            best_t = pop[fits.index(mf)].copy()

        if g%25==0:
            pct = 100*best_f/(MAX_T)
            print(f"  Gen {g:3d}: mejor = {best_f:.1f} pasos ({pct:.0f}% del maximo)")

        if best_f >= MAX_T*0.95*3:
            print(f"  Gen {g}: CONTROL EXCELENTE!")
            break

        tf=sum(fits)+1e-10;probs=[f/tf for f in fits]
        srt=sorted(zip(fits,pop),key=lambda x:x[0],reverse=True)
        new_pop=[srt[0][1].copy(),srt[1][1].copy()]

        while len(new_pop)<pop_size:
            p1=pop[roulette(probs)];p2=pop[roulette(probs)]
            if random.random()<pc: c1,c2=crossover(p1,p2)
            else: c1,c2=p1.copy(),p2.copy()
            if random.random()<pm: c1=mutate(c1)
            if random.random()<pm: c2=mutate(c2)
            if c1.count_nodes()<80: new_pop.append(c1)
            if c2.count_nodes()<80 and len(new_pop)<pop_size: new_pop.append(c2)
        pop=new_pop
    return best_t, best_f, history

# ============================================================
# EJECUCIÓN
# ============================================================
if __name__=="__main__":
    random.seed(42)
    print("="*60)
    print("EJERCICIO 4: PG - CONTROL DE PENDULO INVERTIDO")
    print("="*60)
    print("\nSistema: Pendulo invertido sobre carro (benchmark clasico IA)")
    print(f"  Masa carro   = {mc} kg")
    print(f"  Masa pendulo = {mp} kg")
    print(f"  Longitud     = {L} m")
    print(f"  Max pasos    = {MAX_T} (~{MAX_T*dt:.0f} segundos)")
    print(f"  Falla si: |x|>{X_LIMIT}m o |theta|>{math.degrees(THETA_LIMIT):.0f} grados")

    print("\nConjunto de Terminales T:")
    print("  pos_x, vel_x, angle, ang_vel, sin_angle, cos_angle, 1.0, -1.0, 0.5, -0.5")

    print("\nConjunto de Funciones F:")
    print("  ADD, SUB, MUL, DIV, IFLTE (if<=then/else), NEG, ABS, MAX, MIN")

    print("\nFuncion de Aptitud:")
    print("  f_apt = promedio de pasos en equilibrio sobre 3 condiciones iniciales")
    print("  f_apt optima = MAX_T*3 = " + str(MAX_T*3))

    best_tree, best_fit, hist = run_gp_control(
        pop_size=60, gens=100, pc=0.7, pm=0.02)

    pct = 100*best_fit/(MAX_T*3)*100 if best_fit>0 else 0
    print(f"\n--- MEJOR CONTROLADOR ENCONTRADO ---")
    print(f"f_apt = {best_fit:.1f} (maximo teorico = {MAX_T*3})")
    print(f"Expresion PG del controlador:")
    expr = best_tree.to_string()
    if len(expr)>120: expr=expr[:117]+"..."
    print(f"  Force = {expr}")

    # Simulación detallada del mejor controlador
    print(f"\n--- SIMULACION DETALLADA (condicion inicial: theta=0.05 rad) ---")
    state = [0.0, 0.0, 0.05, 0.0]
    steps_survived = 0
    print(f"{'Paso':>5} | {'x (m)':>8} | {'theta (deg)':>11} | {'Fuerza (N)':>10}")
    print("-"*45)
    for step in range(MAX_T):
        x,xd,th,thd = state
        env = {'pos_x':x,'vel_x':xd,'angle':th,'ang_vel':thd,
               'sin_angle':math.sin(th),'cos_angle':math.cos(th),
               '1.0':1.0,'-1.0':-1.0,'0.5':0.5,'-0.5':-0.5}
        force = eval_tree(best_tree, env)
        force = max(-10.0, min(10.0, force))
        state = step_physics(state, force)
        steps_survived += 1
        if step<10 or (step<100 and step%10==0) or step%50==0:
            print(f"{step:>5} | {state[0]:>8.3f} | {math.degrees(state[2]):>11.3f} | {force:>10.3f}")
        if is_failed(state):
            print(f"\n  !!! FALLO en paso {step} !!!")
            break
    else:
        print(f"\n  Pendulo ESTABLE durante todos los {MAX_T} pasos!")
    print(f"\nPasos en equilibrio: {steps_survived} / {MAX_T}")
    t_survived = steps_survived*dt
    print(f"Tiempo de equilibrio: {t_survived:.1f} segundos")
    perf = "EXCELENTE" if steps_survived==MAX_T else ("BUENO" if steps_survived>MAX_T*0.7 else "PARCIAL")
    print(f"Calificacion: {perf}")